In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *

import os
import time


In [9]:
# input_dir = "x-24-us-election/"

# for root, dirs, files in os.walk(input_dir):
#     for file_name in files:
#         if file_name.endswith(".csv.gz"):
#             # if the file is already decompressed, skip it:
#             if os.path.exists(os.path.splitext(os.path.join(root, file_name))[0]):
#                 print(f"Skipping {file_name} in {root} as it is already decompressed.")
#                 break
#             else:
#                 # decompress the file
#                 print(f"Decompressing {file_name} in {root}")
#                 full_path = os.path.join(root, file_name)
#                 with gzip.open(full_path, 'rt') as f_in:
#                     out_path = os.path.splitext(full_path)[0] 
#                     with open(out_path, 'w') as f_out:
#                         f_out.write(f_in.read())
#                         file_name = f_out.name
#                         print(file_name)

In [10]:

# input_dir = "x-24-us-election/"

# # remove all csv files in input_dir

# for root, dirs, files in os.walk(input_dir):
#     for file_name in files:
#         if file_name.endswith(".csv"):
#             file_name = os.path.join(root, file_name)
#             print(f"Removing {file_name}")
#             os.remove(file_name)


In [11]:
# file_path = "x-24-us-election/part_1/may_july_chunk_1.csv"

# # Read the .csv.gz file into a Spark DataFrame
# df = pd.read_csv(file_path)


# display(df)

In [12]:

# # Initialize Spark session
# spark = SparkSession.builder \
#     .appName("Process Single CSV.GZ") \
#     .config("spark.driver.memory", "4g") \
#     .getOrCreate()

# file_path = "x-24-us-election/part_1/may_july_chunk_1.csv.gz"


# df = spark.read.csv(file_path, header=True, encoding="UTF-8", inferSchema=True, multiLine=True, quote='"', escape='"', mode="PERMISSIVE")

# df.printSchema()

# df.show(20, truncate=False)
# df = df.toPandas()

# df.to_csv("processed_may_july_chunk_1.csv", index=False)


In [13]:
# Initialize Spark session
spark = SparkSession.builder \
    .appName("create dataframes with Spark") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

input_dir = "x-24-us-election/"

# Initialize an empty list to hold Spark DataFrames
dataframes = []

schema = StructType([
    StructField("", IntegerType(), False),
    StructField("id", StringType(), False),
    StructField("text", StringType(), False),
    StructField("url", StringType(), True),
    StructField("epoch", StringType(), True),
    StructField("media", StringType(), True),
    StructField("retweetedTweet", StringType(), True),
    StructField("retweetedTweetID", StringType(), True),
    StructField("retweetedUserID", StringType(), True),
    StructField("id_str", StringType(), True),
    StructField("lang", StringType(), True),
    StructField("rawContent", StringType(), True),
    StructField("replyCount", IntegerType(), True),
    StructField("retweetCount", IntegerType(), True),
    StructField("likeCount", IntegerType(), True),
    StructField("quoteCount", IntegerType(), True),
    StructField("conversationId", StringType(), True),
    StructField("conversationIdStr", StringType(), True),
    StructField("hashtags", StringType(), True),
    StructField("mentionedUsers", StringType(), True),
    StructField("links", StringType(), True),
    StructField("viewCount", IntegerType(), True),
    StructField("quotedTweet", StringType(), True),
    StructField("in_reply_to_screen_name", StringType(), True),
    StructField("in_reply_to_status_id_str", StringType(), True),
    StructField("in_reply_to_user_id_str", StringType(), True),
    StructField("location", StringType(), True),
    StructField("cash_app_handle", StringType(), True),
    StructField("user", StringType(), True),
    StructField("date", StringType(), True),
    StructField("_type", StringType(), True),
    StructField("type", StringType(), True)
])

# Recursively traverse the directory structure
for root, dirs, files in os.walk(input_dir):
    for file_name in files:
        if file_name.endswith(".csv.gz"):
            input_path = os.path.join(root, file_name)
            print(f"Processing file: {input_path}")
            
            df = spark.read.csv(input_path, header=True, 
                                encoding="UTF-8", schema=schema, 
                                multiLine=True, quote='"', escape='"', 
                                mode="PERMISSIVE")
            
            # Append the Spark DataFrame to the list
            dataframes.append(df)

# Combine all Spark DataFrames into a single DataFrame
combined_df = dataframes[0]

total = len(dataframes)
if dataframes:
    for df in dataframes[1:]:
        print(f"Combining DataFrame {dataframes.index(df) + 1} of {total}")
        start = time.time()
        combined_df = combined_df.unionByName(df)
        end = time.time()
        print(f"Combined DataFrame. Time taken: {end - start} seconds")
    print("Combined Spark DataFrame:")
    combined_df.show(5, truncate=False)
    
    # Print the schema of the combined DataFrame
    print("Schema of Combined Spark DataFrame:")
    combined_df.printSchema()
else:
    print("No .csv files found to process.")

Processing file: x-24-us-election/part_1\may_july_chunk_1.csv.gz
Processing file: x-24-us-election/part_1\may_july_chunk_10.csv.gz
Processing file: x-24-us-election/part_1\may_july_chunk_11.csv.gz
Processing file: x-24-us-election/part_1\may_july_chunk_12.csv.gz
Processing file: x-24-us-election/part_1\may_july_chunk_13.csv.gz
Processing file: x-24-us-election/part_1\may_july_chunk_14.csv.gz
Processing file: x-24-us-election/part_1\may_july_chunk_15.csv.gz
Processing file: x-24-us-election/part_1\may_july_chunk_16.csv.gz
Processing file: x-24-us-election/part_1\may_july_chunk_17.csv.gz
Processing file: x-24-us-election/part_1\may_july_chunk_18.csv.gz
Processing file: x-24-us-election/part_1\may_july_chunk_19.csv.gz
Processing file: x-24-us-election/part_1\may_july_chunk_2.csv.gz
Processing file: x-24-us-election/part_1\may_july_chunk_20.csv.gz
Processing file: x-24-us-election/part_1\may_july_chunk_3.csv.gz
Processing file: x-24-us-election/part_1\may_july_chunk_4.csv.gz
Processing fil

In [1]:
sdf = combined_df.select(
    "id", "text", "retweetedTweet", "lang", "replyCount", "retweetCount", "likeCount", "quoteCount", "hashtags", "viewCount", "date")

sdf.show(5, truncate=False)

NameError: name 'combined_df' is not defined

In [ ]:
# order combined_df
combined_df = combined_df.orderBy("date", descending=True)